# E4 S3 Multi-Trait · Native Learnable Code

Pilih **GPU** di Google Colab dan jalankan cell berurutan. Saat diminta, hubungkan Google Drive. Pada sesi pertama, unggah `data_cleaned_3sekolah.xlsx` agar notebook menyimpannya di `MyDrive/AES/datasets/semi_final/`. Sesi berikutnya langsung memakai salinan Drive setelah memeriksa SHA-256 workbook. Folder ini menjadi lokasi dataset semi-final untuk eksperimen berikutnya.

- **Masukan encoder:** pertanyaan, kunci jawaban, jawaban. Email dipakai untuk pembagian kelompok. Jawaban duplikat juga dikelompokkan.
- **Empat keluaran:** isi (1–4), organisasi (1–2), kebahasaan (1–2), mekanik (1–2). Rentang awal mengikuti kategori pada sampel. Cocokkan rentang di cell Pemetaan dengan rubrik resmi sebelum run penelitian. Target diambil langsung dari empat kolom rubrik.
- **Backbone E1 (resep E43):** RoBERTa pretrained dilatih ulang dengan LR 5e-6, LR layer lain 1e-4, Scalar Mix empat layer + gamma, BiGRU, co-attention satu head, masked mean, dan BGE-M3 frozen. MSE merata-ratakan error empat target pada skala 0–1.
- **Validasi E4** memakai lima fold development. Epoch terbaik tiap fold memakai RMSE pada skala skor asli. Untuk memilih satu epoch bagi model bersama, hitung akar dari rata-rata kuadrat error seluruh pasangan skor jawaban × trait. Median epoch terbaik yang dibulatkan ke atas dipakai untuk satu refit pada seluruh development. Evaluasi test dilakukan setelah refit. Seed tetap mengikuti E43 asli.
- **Cara baca kode:** heading kecil menjelaskan fungsi dan kelas. Heading yang menyebut tahap seperti **Hitung fitur BGE**, **Jalankan lima fold**, dan **Evaluasi test** masing-masing punya cell yang benar-benar menjalankan proses serta menampilkan hasilnya.
- **Hasil** berupa Excel per trait, prediksi validasi dan test, serta checkpoint final. RMSE/Pearson memakai prediksi mentah, QWK/Gwet AC2 memakai kategori terdekat. Excel dan checkpoint tetap di runtime. Unduhan akhir mengirim Excel ke komputer. Hasil fold selesai bisa dipakai ulang selama filesystem runtime masih tersedia. Satu fit yang terputus dijalankan ulang.

**Perbandingan E3 → E4.** E3 adalah acuan aktif dengan dropout MLP 0,2. E4 menambahkan sigmoid setelah layer linear terakhir pada head bersama `1024 → 256 → 4`. Sigmoid diterapkan saat training dan inferensi. Tiap keluaran berada pada rentang 0–1, lalu dikembalikan ke skala isi 1–4, organisasi 1–2, kebahasaan 1–2, mekanik 1–2. Loss tetap MSE target normalisasi; pemilihan epoch tetap memakai RMSE validasi skala asli.

Keluaran sigmoid digunakan dalam [implementasi CTS](https://github.com/robert1ridley/cross-prompt-trait-scoring/blob/HEAD/models/CTS.py) dan [ProTACT](https://github.com/doheejin/ProTACT/blob/HEAD/models/ProTACT.py). E4 menguji adaptasi aktivasi ini pada S3. Checkpoint dan hasil E4 mempunyai folder run tersendiri. Bandingkan RMSE validasi dan hasil per trait dengan E3.


## 1 · Setup pustaka pada kernel Colab

In [ ]:
# Jalankan pada runtime Colab baru, sebelum mengimpor pustaka numerik/model.
import sys
import subprocess
from importlib import metadata, util
from packaging.requirements import Requirement

# Pertahankan Torch/CUDA, NumPy, pandas, SciPy dan sklearn bawaan Colab.
# Mengganti pustaka numerik yang sudah dimuat kernel dapat menyebabkan konflik ABI.
CORE_REQUIREMENTS = {
    'torch': 'torch>=2.6', 'numpy': 'numpy>=1.26,<3',
    'pandas': 'pandas>=2,<3', 'scipy': 'scipy>=1.10,<2',
    'scikit-learn': 'scikit-learn>=1.2,<2',
}
for package, requirement in CORE_REQUIREMENTS.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError('Gunakan runtime Google Colab baru; pustaka dasar belum ada: '+package) from error
    if not Requirement(requirement).specifier.contains(installed):
        raise RuntimeError(requirement+' diperlukan; runtime ini memiliki '+installed+'. Pilih runtime Colab baru.')

# Tokenizers 0.19 tidak menyediakan wheel Python 3.13; tetap satu versi per runtime.
TRANSFORMERS_REQUIREMENT = 'transformers==4.48.3' if sys.version_info >= (3, 13) else 'transformers==4.44.2'
EXTRA_REQUIREMENTS = [TRANSFORMERS_REQUIREMENT, 'openpyxl>=3.1,<4']
to_install = []
for requirement in EXTRA_REQUIREMENTS:
    req = Requirement(requirement)
    try:
        installed = metadata.version(req.name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not req.specifier.contains(installed):
        if req.name in sys.modules:
            raise RuntimeError(req.name+' sudah diimpor dengan versi berbeda. Restart session, lalu Run all dari atas.')
        to_install.append(requirement)
if to_install:
    # Kendala ini mencegah pip mengganti pustaka numerik yang digunakan kernel.
    from pathlib import Path
    constraints = Path('/tmp/aes35_native_constraints.txt')
    constraints.write_text(''.join(p+'=='+metadata.version(p)+'\n' for p in CORE_REQUIREMENTS))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           '--constraint', str(constraints), *to_install])
print('Setup selesai. Semua cell berikutnya berjalan langsung di kernel notebook ini.')
print('Python:', sys.version.split()[0])


## 2 · Import dan ruang variabel notebook

In [ ]:
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging, html
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung PEFT.
    # Backend teks dikonfigurasi pada kernel notebook; tidak memuat operator vision.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
SCENARIOS = ('S3',)  # Hanya model fusion yang dijalankan.
NOTEBOOK_CODE_SHA256='f875bd85a4057c573590eb955128bada9d0e6bdd76d01c111f476ee97a077f32'

from IPython.display import display, HTML


## 3 · Nama model pretrained

In [ ]:
# NAMA MODEL PRETRAINED — ubah di sini jika membuat eksperimen lain.
ROBERTA_CHECKPOINT = 'flax-community/indonesian-roberta-base'
BGE_CHECKPOINT = 'BAAI/bge-m3'

# Revision mengunci bobot dan tokenizer yang sama seperti E30.
ROBERTA_REVISION = 'cc5ecee7de8532880ad08d802292a6f42865bc83'
BGE_REVISION = '5617a9f61b028005a4858fdac845db406aefb181'

print('RoBERTa:', ROBERTA_CHECKPOINT, '| encoder ikut fine-tuning')
print('BGE-M3:', BGE_CHECKPOINT, '| encoder frozen; proyeksi/head tetap belajar')


## 4 · Pemetaan input dan empat target

In [ ]:
SCOPE = 'e43_s3_multitrait_three_schools'
SHEET_NAME = 'Sheet1'
QUESTION_COLUMN = 'Pertanyaan'
REFERENCE_COLUMN = 'Kunci_Jawaban'
ANSWER_COLUMN = 'Jawaban'
GROUP_COLUMN = 'Email Address'
RESPONSE_NUMBER_COLUMN = 'No_Soal'
INPUT_COLUMNS = (QUESTION_COLUMN, REFERENCE_COLUMN, ANSWER_COLUMN)

# Setel sesuai pedoman penilaian sebelum training. Rentang awal sesuai kategori di file contoh.
TRAITS = (
    ('isi', 'Skor_Isi', 1, 4),
    ('organisasi', 'Skor_Organisasi', 1, 2),
    ('kebahasaan', 'Skor_Tata_Bahasa', 1, 2),
    ('mekanik', 'Skor_Mekanik', 1, 2),
)
TARGET_COLUMNS = tuple(column for _, column, _, _ in TRAITS)
REQUIRED = (*INPUT_COLUMNS, GROUP_COLUMN, RESPONSE_NUMBER_COLUMN, *TARGET_COLUMNS)
RUBRIC = {'traits': [
    {'name': name, 'min': low, 'max': high, 'categories': list(range(low, high + 1))}
    for name, _, low, high in TRAITS
]}
SCORE_MIN = np.asarray([low for _, _, low, _ in TRAITS], dtype=np.float32)
SCORE_SPAN = np.asarray([high - low for _, _, low, high in TRAITS], dtype=np.float32)


## 5 · Hyperparameter E4

In [ ]:
# HIPERPARAMETER BASELINE E43; Scalar Mix empat layer terakhir.
EPOCHS = 25
FOLDS = 5
EARLY_STOPPING_PATIENCE = 5
TRAIN_SEED = 42  # Satu konstanta pengacakan, tanpa loop multi-seed.
SPLIT_SEED = 2026
TEST_FRACTION = 0.20
MICRO_BATCH = 1
EFFECTIVE_BATCH = 8

ENCODER_LR = 5e-6             # Tetap E30; bukan LR E37 7e-6.
S3_MLP_LR = 1e-4              # MLP S3 tetap nilai asli E30.
S3_LAYERS_LR = 1e-4           # Kembali ke E30: BiGRU/co-attention/pooling/norm = 1e-4.
PROJECTION_LR = 1e-4          # Proyeksi BGE projection.* tidak ikut diturunkan.

BIGRU_HIDDEN_PER_DIRECTION = 64   # E30: dua arah menghasilkan128fitur/token.
POOLING_ATTENTION_HIDDEN = 64  # Ukuran init acuan; layer skor attention dihapus sebelum training.
DROPOUT = 0.1                   # Parameter lama untuk layer lain; tetap E30.
S3_MLP_DROPOUT = 0.2             # E4: dropout MLP 0,2 mengikuti E3.
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRADIENT_CLIP = 1.0

ROBERTA_MAX_TOKENS = 512
BGE_MAX_TOKENS = 1024
WINDOW_OVERLAP = 128
CONTEXT_MAX_TOKENS = 192
QUESTION_MAX_TOKENS = 64
DEVICE = 'cuda'

SCALAR_MIX_LAST_LAYERS = 4      # Empat layer Transformer terakhir, tidak termasuk embeddings.
SCALAR_MIX_LR = 1e-4            # LR untuk empat logit campuran yang dipelajari.

SCALAR_MIX_GAMMA_INITIAL = 1.0  # Satu parameter baru; LR mengikuti SCALAR_MIX_LR=1e-4.


### `config`

Periksa konfigurasi model, ukuran batch, dan anggaran token.

In [ ]:
# Nilai bawaan dan fungsi konfigurasi, tersedia langsung di notebook.

DEFAULTS = {
    'sheet': SHEET_NAME, 'expected_rows': None,
    'test_fraction': TEST_FRACTION, 'folds': FOLDS, 'seed': TRAIN_SEED, 'split_seed': SPLIT_SEED, 'split_candidates': 128,
    # Maksimal 25 epoch/fold; berhenti bila RMSE validasi tidak membaik selama patience.
    'epochs': EPOCHS, 'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'effective_batch': EFFECTIVE_BATCH, 'micro_batch': MICRO_BATCH,
    'encoder_lr': ENCODER_LR, 's3_encoder_lr': ENCODER_LR, 'projection_lr': PROJECTION_LR, 'head_lr': 1e-4, 'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO, 'gradient_clip': GRADIENT_CLIP, 'dropout': DROPOUT, 's3_mlp_dropout': S3_MLP_DROPOUT,
    'roberta_max_length': ROBERTA_MAX_TOKENS, 'roberta_overlap': WINDOW_OVERLAP,
    'bge_max_length': BGE_MAX_TOKENS, 'bge_overlap': WINDOW_OVERLAP, 'bge_batch': 4, 'device': DEVICE,
    'roberta': {'checkpoint': ROBERTA_CHECKPOINT,
                'revision': ROBERTA_REVISION},
    'bge': {'checkpoint': BGE_CHECKPOINT,
            'revision': BGE_REVISION},
    # Batas konteks dihitung termasuk prefix; semua token jawaban tetap dicakup.
    'context_max_tokens': CONTEXT_MAX_TOKENS, 'question_max_tokens': QUESTION_MAX_TOKENS,
    'new_question': '', 'new_reference': '',
    'input_policy':'three_texts_all_scenarios_v1', 'input_format':'single_sequence',
    's3_pooling': 'bigru_coattention_mean_pooling', 's3_sequence_hidden': BIGRU_HIDDEN_PER_DIRECTION, 'pooling_attention_hidden': POOLING_ATTENTION_HIDDEN,
    'scalar_mix_gamma_init': SCALAR_MIX_GAMMA_INITIAL, 'scalar_mix_layers': SCALAR_MIX_LAST_LAYERS, 'scalar_mix_lr': SCALAR_MIX_LR,
    's3_mlp_lr': S3_MLP_LR, 's3_layers_lr': S3_LAYERS_LR,
    's3_fusion': 'concat', 
    'vector_source': 'pretrained', 'trace_position': 0,
}


# Gabungkan parameter pilihan dengan nilai bawaan, lalu periksa batas skor, batch, dan panjang input.
def config(values):
    result = {**DEFAULTS, **values}
    if not 0 < result['test_fraction'] < 1 or result['folds'] < 2 or result['split_candidates'] < 1:
        raise ValueError('Split/folds tidak sah')
    if result['epochs'] < 1 or not 1 <= result['micro_batch'] <= result['effective_batch']:
        raise ValueError('Epoch/batch tidak sah')
    if type(result['early_stopping_patience']) is not int or result['early_stopping_patience'] < 1:
        raise ValueError('early_stopping_patience harus bilangan bulat positif.')
    if result['vector_source'] not in ('pretrained', 'pretrained_pair', 'S3'):
        raise ValueError('vector_source harus pretrained, pretrained_pair, atau S3')
    if result['device'] not in ('auto', 'cpu', 'cuda'):
        raise ValueError('device harus auto, cpu, atau cuda')
    if not 0 <= result['warmup_ratio'] < 1:
        raise ValueError('warmup_ratio tidak sah')
    if any(not math.isfinite(result[k]) or result[k] <= 0
           for k in ('encoder_lr', 's3_encoder_lr', 'projection_lr', 'head_lr', 's3_mlp_lr', 's3_layers_lr', 'gradient_clip', 'bge_batch')):
        raise ValueError('Learning rate, gradient clip, dan batch BGE harus positif.')
    if not 0 <= result['s3_mlp_dropout'] < 1:
        raise ValueError('s3_mlp_dropout harus berada pada rentang [0,1).')
    if not 0 <= result['dropout'] < 1 or result['weight_decay'] < 0:
        raise ValueError('Dropout/weight decay tidak sah.')
    if not 2 < result['roberta_max_length'] <= 512 or not 0 <= result['roberta_overlap'] < result['roberta_max_length']-2:
        raise ValueError('Window RoBERTa melebihi batas atau overlap tidak sah.')
    if not 2 < result['bge_max_length'] <= 8192:
        raise ValueError('Batas BGE harus 3–8192 token termasuk special token.')
    if not (type(result['context_max_tokens']) is int and type(result['question_max_tokens']) is int
            and 1 <= result['question_max_tokens'] < result['context_max_tokens']
            and result['context_max_tokens']+4+result['roberta_overlap'] < result['roberta_max_length']):
        raise ValueError('Anggaran konteks harus menyisakan token jawaban lebih banyak daripada overlap.')
    if str(result.get('new_answer', '')).strip() and not all(str(result.get(k, '')).strip() for k in ('new_question', 'new_reference')):
        raise ValueError('Untuk NEW_ANSWER baru, isi juga NEW_QUESTION dan NEW_REFERENCE.')
    if not (type(result['bge_overlap']) is int and 0 <= result['bge_overlap']
            < result['bge_max_length']-result['context_max_tokens']-4):
        raise ValueError('Anggaran konteks BGE harus menyisakan kapasitas jawaban di atas overlap.')
    if result['input_policy']!='three_texts_all_scenarios_v1':
        raise ValueError('Notebook ini khusus tiga input pada semua skenario.')
    if result['input_format'] not in ('single_sequence','context_pair'):
        raise ValueError('Format input tidak sah.')
    if result['s3_pooling'] != 'bigru_coattention_mean_pooling' or type(result['s3_sequence_hidden']) is not int or result['s3_sequence_hidden'] < 1:
        raise ValueError('Jenis/hidden pooling S3 tidak cocok notebook eksperimen ini.')
    if type(result['pooling_attention_hidden']) is not int or result['pooling_attention_hidden'] < 1:
        raise ValueError('pooling_attention_hidden harus bilangan bulat positif.')
    if type(result['scalar_mix_layers']) is not int or result['scalar_mix_layers'] < 1:
        raise ValueError('scalar_mix_layers harus bilangan bulat positif.')
    if not math.isfinite(result['scalar_mix_lr']) or result['scalar_mix_lr'] <= 0:
        raise ValueError('scalar_mix_lr harus positif dan finite.')
    if isinstance(result['scalar_mix_gamma_init'], bool) or not math.isfinite(result['scalar_mix_gamma_init']):
        raise ValueError('scalar_mix_gamma_init harus finite.')
    if result['s3_fusion'] != 'concat':
        raise ValueError('Fusion/hidden gate tidak cocok dengan notebook ini.')
    return result


### `device_name`

Tentukan perangkat untuk encoder dan training.

In [ ]:
def device_name(cfg):
    requested = cfg['device']
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA diminta tetapi tidak tersedia; pilih GPU di Colab atau device=cpu.')
    return ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested


## 6 · Dataset semi-final di Google Drive

### `read_dataset` dan `Bundle`

Workbook dibaca sebagai tiga kolom teks dan matriks empat skor. Email hanya dipakai untuk membentuk kelompok split.

In [ ]:
# Baca dan audit data tanpa mengubah file Excel asli.
# Rapikan Unicode dan karakter kontrol; ejaan, huruf besar, dan tanda baca jawaban tetap dipertahankan.
def clean(text, normalize_punctuation=False):
    text = unicodedata.normalize('NFC', text).replace('\r\n','\n').replace('\r','\n')
    text = ''.join(c for c in text if c in '\n\t' or unicodedata.category(c) != 'Cc')
    if normalize_punctuation:
        text = text.translate(str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-'})).replace('…','...')
    return text

def digest(path):
    # Checkpoint besar diperiksa bertahap supaya tidak dimuat seluruhnya ke RAM.
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


@dataclass
class Bundle:
    rows: list
    y: np.ndarray
    groups: np.ndarray
    source_sha256: str
    summary: dict

    @property
    def texts(self):
        return [r['text'] for r in self.rows]


# Baca jawaban dan nilai guru, periksa kelengkapan, lalu kelompokkan email/duplikat untuk mencegah kebocoran data.
def read_dataset(cfg):
    path = Path(cfg['dataset'])
    source_hash = digest(path)
    book = load_workbook(path, read_only=True, data_only=True)
    try:
        if cfg['sheet'] not in book.sheetnames:
            raise ValueError(f"Tab {cfg['sheet']} tidak tersedia: {book.sheetnames}")
        data = iter(book[cfg['sheet']].iter_rows(values_only=True))
        header = [str(x).strip() if x is not None else None for x in next(data, ())]
        if any(header.count(x) != 1 for x in REQUIRED):
            raise ValueError('Header wajib hilang atau duplikat: '+', '.join(REQUIRED))
        rows = []
        for excel_row, values in enumerate(data, 2):
            if all(v is None for v in values):
                continue  # Hanya baris format kosong, bukan label/jawaban yang hilang.
            raw = {k: values[i] for i, k in enumerate(header) if k is not None}
            missing = [k for k in REQUIRED if raw[k] is None or not str(raw[k]).strip()]
            if missing:
                raise ValueError(f'Baris {excel_row}: kolom wajib kosong: '+', '.join(missing))
            scores = []
            for _, column, low, high in TRAITS:
                score = raw[column]
                if (isinstance(score, bool) or not isinstance(score, (int, float))
                    or not math.isfinite(score) or int(score) != score or not low <= score <= high):
                    raise ValueError(f'Baris {excel_row}: {column} harus bilangan bulat {low}–{high}.')
                scores.append(int(score))
            for key in (QUESTION_COLUMN,REFERENCE_COLUMN):
                if not isinstance(raw[key],str) or not raw[key].strip():
                    raise ValueError(f'Baris {excel_row}: {key} harus teks yang tidak kosong.')
            if not isinstance(raw[ANSWER_COLUMN], str) or not clean(raw[ANSWER_COLUMN]).strip():
                raise ValueError(f'Baris {excel_row}: jawaban bukan teks.')
            rows.append({'row_id': f'row_{excel_row-1:04d}', 'source_row': excel_row,
                         'text': clean(raw[ANSWER_COLUMN]),
                         'email_group': unicodedata.normalize('NFC', str(raw[GROUP_COLUMN])).strip().casefold(),
                         'question': clean(raw[QUESTION_COLUMN]), 'reference': clean(raw[REFERENCE_COLUMN]),
                         'response_number': str(raw[RESPONSE_NUMBER_COLUMN]), 'scores': scores})
    finally:
        book.close()
    if cfg.get('expected_rows') is not None and len(rows) != cfg['expected_rows']:
        raise ValueError(f"Diharapkan {cfg['expected_rows']} baris, ditemukan {len(rows)}; tidak dipotong otomatis.")
    if not rows:
        raise ValueError('Dataset kosong')
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    email_first, duplicate_sets = {}, {}
    for i, row in enumerate(rows):
        email = row['email_group']
        if email in email_first:
            union(email_first[email], i)
        email_first[email] = i
        # Normalisasi ini HANYA audit kesamaan, bukan teks masukan encoder.
        normalized = ' '.join(row['text'].casefold().split())
        key = hashlib.sha256(normalized.encode()).hexdigest()
        duplicate_sets.setdefault(key, []).append(i)
    for indices in duplicate_sets.values():
        for i in indices[1:]:
            union(indices[0], i)
    roots = {root: i for i, root in enumerate(sorted({find(i) for i in range(len(rows))}))}
    groups = np.asarray([roots[find(i)] for i in range(len(rows))], dtype=int)
    y = np.asarray([row.pop('scores') for row in rows], dtype=np.float32)
    summary = {
        'scope': SCOPE, 'rows': len(rows), 'target_columns': list(TARGET_COLUMNS),
        'model_outputs': len(TRAITS), 'categories': RUBRIC['traits'],
        'email_groups': len(email_first), 'connected_groups': len(roots),
        'questions': len({row['response_number'] for row in rows}),
        'duplicate_sets': sum(len(indices) > 1 for indices in duplicate_sets.values()),
        'source_sha256': source_hash, 'input_features': list(INPUT_COLUMNS),
    }
    for row in rows:
        del row['email_group']
    return Bundle(rows, y, groups, source_hash, summary)

# Identitas hasil mengikuti data, konfigurasi, dan kode notebook ini.
def run_folder(cfg, bundle):
    code_hash = NOTEBOOK_CODE_SHA256
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    key = hashlib.sha256(json.dumps({'source': bundle.source_sha256, 'config': semantic, 'code': code_hash}, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path(cfg['output']) / key
    folder.mkdir(parents=True, exist_ok=True)
    dump(folder / 'effective_config.json', cfg)
    return folder


In [ ]:
# Notebook eksperimen berikutnya bisa memakai path dan SHA-256 yang sama.
DATASET_VERSION = 'semi-final'
DATASET_NAME = 'data_cleaned_3sekolah.xlsx'
DATASET_SHA256 = 'dfbd97128b09a4eb488dd890ccc3efd9b9fe232e74fcc00ee5504a03acc58226'
DRIVE_MOUNT = Path('/content/drive')
DRIVE_DATASET = DRIVE_MOUNT / 'MyDrive' / 'AES' / 'datasets' / 'semi_final' / DATASET_NAME
LOCAL_DATASET = Path(DATASET_NAME)  # Untuk eksekusi di luar Colab.
OUTPUT_ROOT = Path('outputs/aes_e43_multitrait')


def resolve_colab_dataset(mount_drive, upload, drive_mount, drive_dataset):
    """Gunakan salinan privat Drive atau simpan workbook yang cocok pada upload pertama."""
    from tempfile import NamedTemporaryFile

    mount_drive(str(drive_mount))
    if not (drive_mount / 'MyDrive').is_dir():
        raise RuntimeError('MyDrive belum tersedia. Hubungkan Google Drive di Colab.')
    if not drive_dataset.exists():
        print(f'Unggah {DATASET_NAME} untuk penyimpanan pertama di Drive.')
        uploads = upload()
        if len(uploads) != 1:
            raise ValueError(f'Unggah satu workbook asli bernama {DATASET_NAME}.')
        workbook = next(iter(uploads.values()))
        if hashlib.sha256(workbook).hexdigest() != DATASET_SHA256:
            raise ValueError('File unggahan berbeda dari workbook semi-final yang dipakai eksperimen ini.')
        drive_dataset.parent.mkdir(parents=True, exist_ok=True)
        with NamedTemporaryFile(dir=drive_dataset.parent, suffix='.pending', delete=False) as stream:
            staging = Path(stream.name)
            try:
                stream.write(workbook)
                stream.close()
                if digest(staging) != DATASET_SHA256:
                    raise IOError('Salinan di Drive gagal diverifikasi.')
                if not drive_dataset.exists():
                    staging.replace(drive_dataset)
            finally:
                staging.unlink(missing_ok=True)
        print(f'Workbook semi-final tersedia di Drive: {drive_dataset}')
    return drive_dataset


if 'google.colab' in sys.modules or Path('/content').is_dir():
    from google.colab import drive, files
    DATASET_PATH = resolve_colab_dataset(drive.mount, files.upload, DRIVE_MOUNT, DRIVE_DATASET)
else:
    DATASET_PATH = LOCAL_DATASET

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Workbook semi-final belum ditemukan: {DATASET_PATH}')
if digest(DATASET_PATH) != DATASET_SHA256:
    raise ValueError(f'Workbook di {DATASET_PATH} berbeda dari versi semi-final terpilih. '
                     'Gunakan file asli atau tetapkan path dan SHA-256 versi baru secara eksplisit.')
print(f'Dataset {DATASET_VERSION} terverifikasi: {DATASET_PATH}')


## 7 · Baca workbook dan periksa X serta y

In [ ]:
cfg = config({
    'dataset': str(DATASET_PATH), 'output': str(OUTPUT_ROOT),
    'experiment': 'E4', 'output_activation': 'sigmoid', 'dataset_version': DATASET_VERSION, 'traits': TRAITS,
    'runtime': {'torch': str(torch.__version__), 'transformers': version('transformers'),
                'scikit-learn': version('scikit-learn'), 'device': device_name(DEFAULTS),
                'accelerator': torch.cuda.get_device_name() if DEVICE == 'cuda' else None},
})
bundle = read_dataset(cfg)
print(f"{len(bundle.rows)} jawaban, {bundle.summary['email_groups']} akun, "
      f"{bundle.summary['connected_groups']} kelompok setelah penggabungan duplikat.")
print(f"X berisi {len(INPUT_COLUMNS)} kolom teks: {', '.join(INPUT_COLUMNS)}")
print(f"y berukuran {bundle.y.shape}")
display(pd.DataFrame([{
    'trait': name, 'kolom_target': column, 'minimum': low, 'maksimum': high,
} for name, column, low, high in TRAITS]))


## 8 · Pisahkan development, test, dan lima fold

### `make_split`

Seluruh jawaban seorang siswa dan kelompok duplikat tetap bersama. Lima fold dibuat dari development.

In [ ]:
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest


### `contexts_from_bundle`

Ambil pasangan pertanyaan dan referensi yang mengiringi setiap jawaban.

In [ ]:
def contexts_from_bundle(bundle):
    """Ambil soal/referensi tanpa membaca target skor guru."""
    contexts = []
    for row in bundle.rows:
        question, reference = row.get('question'), row.get('reference')
        if not isinstance(question, str) or not question.strip():
            raise ValueError('Context pair memerlukan soal yang tidak kosong.')
        if not isinstance(reference, str) or not reference.strip():
            raise ValueError('Context pair memerlukan jawaban referensi yang tidak kosong.')
        contexts.append((clean(question), clean(reference)))
    return contexts


In [ ]:
manifest = make_split(bundle, cfg)
output = run_folder(cfg, bundle)
dump(output / 'split.json', manifest)

print(f"Development: {len(manifest['development'])} | Test: {len(manifest['test'])}")
display(pd.DataFrame([{
    'fold': f['fold'], 'training': len(f['train']), 'validation': len(f['validation']),
} for f in manifest['folds']]))


## 9 · Tokenisasi, window, dan posisi input

### `mean_by_essay`

Rata-ratakan vektor window yang berasal dari jawaban yang sama.

In [ ]:
def mean_by_essay(cls, mapping, n):
    sums=cls.new_zeros((n,cls.shape[-1])).index_add(0,mapping,cls)
    counts=cls.new_zeros(n).index_add(0,mapping,cls.new_ones(len(mapping)))
    if (counts==0).any(): raise ValueError('Esai tanpa window')
    return sums / counts[:,None]


### `context_token_ids`

Ubah soal dan referensi jadi ID token sesuai batas konteks.

In [ ]:
def context_token_ids(context, tokenizer, cfg):
    """Sisakan anggaran untuk soal dan referensi sekaligus."""
    if not isinstance(context, (tuple, list)) or len(context) != 2:
        raise ValueError('Konteks harus pasangan (soal, jawaban referensi).')
    question, reference = context
    if not all(isinstance(t, str) and t.strip() for t in context):
        raise ValueError('Isi soal dan jawaban referensi sebelum memprediksi.')
    limit, qlimit = cfg['context_max_tokens'], cfg['question_max_tokens']
    if not 1 <= qlimit < limit:
        raise ValueError('Anggaran token soal harus lebih kecil daripada konteks.')
    # Prefix menjadi bagian input; nilai guru tidak pernah dimasukkan.
    qids = tokenizer.encode('Soal: '+question, add_special_tokens=False)
    rids = tokenizer.encode('\nJawaban referensi: '+reference, add_special_tokens=False)
    question_kept = qids[:qlimit]
    reference_kept = rids[:limit-len(question_kept)]
    return question_kept+reference_kept, {
        'question_tokens': len(qids), 'question_kept': len(question_kept),
        'reference_tokens': len(rids), 'reference_kept': len(reference_kept),
        'question_truncated': len(question_kept) < len(qids),
        'reference_truncated': len(reference_kept) < len(rids),
        'context_tokens_kept': len(question_kept)+len(reference_kept),
    }


### `reference_context_positions`

Tandai posisi token referensi untuk co-attention.

In [ ]:
def reference_context_positions(context, tok, cfg, info):
    """Posisi isi referensi dari offset karakter; ID token berulang tidak ditebak."""
    prefix = '\nJawaban referensi: '
    text = prefix + context[1]
    try:
        aligned = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    except NotImplementedError:
        # RoBERTa training tetap slow seperti E26; fast hanya membaca offset.
        if not hasattr(tok, '_aes_reference_fast'):
            spec = cfg['roberta']
            tok._aes_reference_fast = AutoTokenizer.from_pretrained(
                spec['checkpoint'], revision=spec['revision'], use_fast=True)
        aligned = tok._aes_reference_fast(text, add_special_tokens=False, return_offsets_mapping=True)
    original = tok.encode(text, add_special_tokens=False)
    if list(aligned['input_ids']) != original or len(aligned['offset_mapping']) != len(original):
        raise ValueError('ID/offset tokenizer referensi tidak identik; posisi tidak ditebak.')
    positions = []
    for index, (start, end) in enumerate(aligned['offset_mapping'][:info['reference_kept']]):
        if not 0 <= start <= end <= len(text):
            raise ValueError('Offset tokenizer referensi tidak sah.')
        # Token yang mencakup teks isi diperbolehkan; token prefix/spasi saja dikecualikan.
        if end > len(prefix) and text[max(start, len(prefix)):end].strip():
            positions.append(info['question_kept'] + index)
    if not positions:
        raise ValueError('Isi referensi habis oleh truncation/prefix. Tambah anggaran konteks sebelum training.')
    return positions


### `pair_content_windows`

Rangkai tiga teks dan bagi jawaban panjang menjadi window yang bertumpang tindih.

In [ ]:
def pair_content_windows(text, context, tokenizer, maximum=512, overlap=128, cfg=None):
    """Informasi dan window sama; E22 menambah separator resmi pasangan."""
    if cfg is None or cfg.get('input_format') not in ('single_sequence', 'context_pair'):
        raise ValueError('Pilih format input E21 atau E22 sebelum tokenisasi.')
    context_ids, context_info = context_token_ids(context, tokenizer, cfg)
    reference_positions = reference_context_positions(context, tokenizer, cfg, context_info)
    context_layout = [-1]*len(context_ids)
    for position in reference_positions:context_layout[position] = -4
    ids = tokenizer.encode(text, add_special_tokens=False)
    answer_prefix = tokenizer.encode('\nJawaban siswa:\n', add_special_tokens=False)
    if not ids:
        raise ValueError('Jawaban tidak menghasilkan token.')
    paired = cfg['input_format'] == 'context_pair'
    specials = tokenizer.num_special_tokens_to_add(pair=paired)
    # Cadangkan overhead terbesar untuk KEDUA eksperimen. Starts/cakupan sama.
    reserved = max(tokenizer.num_special_tokens_to_add(pair=False),
                   tokenizer.num_special_tokens_to_add(pair=True))
    capacity = maximum-len(context_ids)-len(answer_prefix)-reserved
    if not 0 <= overlap < capacity:
        raise ValueError('Konteks dan penanda jawaban menyisakan kapasitas di bawah overlap.')
    starts = [0]
    while starts[-1]+capacity < len(ids):
        starts.append(starts[-1]+capacity-overlap)
    chunks, answer_positions, reference_window_positions, segments = [], [], [], []
    for start in starts:
        answer = ids[start:start+capacity]
        second = answer_prefix+answer
        if paired:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids, second)
            layout = tokenizer.build_inputs_with_special_tokens(context_layout,
                     [-3]*len(answer_prefix)+[-2]*len(answer))
        else:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids+second)
            layout = tokenizer.build_inputs_with_special_tokens(
                     context_layout+[-3]*len(answer_prefix)+[-2]*len(answer))
        positions = [i for i, value in enumerate(layout) if value == -2]
        if (len(prepared) != len(layout) or len(prepared) > maximum
                or len(prepared)-len(context_ids)-len(second) != specials
                or [prepared[i] for i in positions] != answer):
            raise ValueError('Template tokenizer tidak konsisten dengan format input.')
        chunks.append({'input_ids':prepared, 'attention_mask':[1]*len(prepared)})
        answer_positions.append(positions)
        reference_window_positions.append([i for i, value in enumerate(layout) if value == -4])
        segments.append(['Soal/referensi' if v == -1 else 'Jawaban siswa' if v == -2
                         else 'Penanda jawaban' if v == -3 else 'Isi referensi' if v == -4 else 'Token khusus' for v in layout])
    stats = {**context_info, 'input_format':cfg['input_format'], 'content_tokens':len(ids),
             'capacity':capacity, 'stride':capacity-overlap, 'starts':starts,
             'windows':len(chunks), 'pair_special_tokens':specials,
             'reserved_special_tokens':reserved, 'answer_prefix_tokens':len(answer_prefix),
             'answer_positions':answer_positions, 'reference_positions':reference_window_positions,
             'reference_content_kept':len(reference_positions), 'segments':segments, 'answer_tokens_uncovered':0}
    return chunks, stats


### `pair_batch_windows`

Tambahkan padding dan simpan pemetaan window ke jawaban.

In [ ]:
def pair_batch_windows(texts, contexts, tokenizer, device='cpu', maximum=512, overlap=128, cfg=None):
    """Susun batch pasangan tanpa menukar konteks antarjawaban."""
    if len(texts) != len(contexts) or not texts:
        raise ValueError('Jumlah konteks harus sama dengan jumlah jawaban.')
    chunks, mapping, stats = [], [], []
    for index, (text, context) in enumerate(zip(texts, contexts)):
        values, detail = pair_content_windows(text, context, tokenizer, maximum, overlap, cfg)
        chunks.extend(values)
        mapping.extend([index]*len(values))
        stats.append(detail)
    batch = tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {key: value.to(device) for key, value in batch.items()}, torch.tensor(mapping, device=device), stats


## 10 · Memuat tokenizer dan encoder pretrained

### `release`

Bebaskan alokasi yang selesai dipakai sebelum berpindah tahap.

In [ ]:
def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
class ExpectedPoolerNotice(logging.Filter):
    """Hanya saring pemberitahuan dua bobot pooler yang memang tidak dipakai."""
    def filter(self, record):
        import ast
        import re
        message = record.getMessage()
        if not message.startswith('Some weights of the model checkpoint at ') or 'were not used when initializing' not in message:
            return True
        match = re.search(r"\['[^\n]+?\]", message)
        if match is None:
            return True
        try:
            keys = ast.literal_eval(match.group())
        except (ValueError, SyntaxError):
            return True
        return set(keys) != {'pooler.dense.bias', 'pooler.dense.weight'}


### `encoder`

Muat bobot pretrained. Jalur BGE dibekukan dan RoBERTa disiapkan untuk fine-tuning.

In [ ]:
def encoder(cfg, role):
    # Ambil nama checkpoint dan revision model yang dipilih: roberta atau bge.
    spec = cfg[role]
    logger = logging.getLogger('transformers.modeling_utils')
    notice = ExpectedPoolerNotice()
    logger.addFilter(notice)
    try:
        # Muat arsitektur dan bobot pretrained. Ini baru inisialisasi encoder, belum latihan memakai nilai guru.
        result = AutoModel.from_pretrained(spec['checkpoint'], revision=spec['revision'], add_pooling_layer=False)
    finally:
        logger.removeFilter(notice)
    # Periksa panjang vektor: RoBERTa 768 angka, BGE-M3 1024 angka.
    expected = 768 if role == 'roberta' else 1024
    if result.config.hidden_size != expected:
        raise ValueError(f'{role}: dimensi checkpoint tidak cocok dengan {expected}.')
    # Parameter RoBERTa ikut diperbarui; parameter BGE tidak ikut diperbarui.
    # RoBERTa boleh belajar; bobot encoder BGE tidak diperbarui.
    result.requires_grad_(role == 'roberta')
    return result


### `tokenizer`

Muat tokenizer dari repositori dan revisi yang cocok dengan encoder.

In [ ]:
def tokenizer(cfg, role, fast=False):
    # Muat tokenizer dari checkpoint yang sama agar ID token cocok dengan kosakata model.
    result = AutoTokenizer.from_pretrained(cfg[role]['checkpoint'], revision=cfg[role]['revision'],
                                           use_fast=fast if role == 'roberta' else True)
    if role == 'roberta' and not fast and result.num_special_tokens_to_add(False) != 2:
        raise ValueError('RoBERTa slow tokenizer harus menambahkan dua special token.')
    return result


### Cek window pada satu jawaban development

Tabel menampilkan panjang input dan jumlah token yang ditandai untuk referensi serta jawaban.

In [ ]:
# Ambil satu jawaban development dan perlihatkan ukuran window serta posisi yang dipakai E43.
example_index = manifest['development'][0]
example_row = bundle.rows[example_index]
example_context = (example_row['question'], example_row['reference'])
example_tokenizer = tokenizer(cfg, 'roberta')
example_windows, example_detail = pair_content_windows(
    example_row['text'], example_context, example_tokenizer,
    cfg['roberta_max_length'], cfg['roberta_overlap'], cfg,
)
display(pd.DataFrame([{
    'id_jawaban': example_row['row_id'],
    'window': n,
    'panjang_input': len(window['input_ids']),
    'token_referensi': len(reference),
    'token_jawaban': len(answer),
} for n, (window, reference, answer) in enumerate(zip(
    example_windows, example_detail['reference_positions'], example_detail['answer_positions'],
), 1)]))


## 11 · Fitur BGE-M3 frozen

### `paired_pretrained_values`

Gabungkan vektor dari semua window menjadi satu vektor per jawaban.

In [ ]:
def paired_pretrained_values(bundle, cfg, role):
    """Mean CLS antar-window; BGE dinormalisasi L2 setelah agregasi."""
    device = device_name(cfg)
    model, tok = encoder(cfg,role).to(device), tokenizer(cfg,role)
    model.eval()
    vectors, window_vectors, ids, coverage = [], [], [], []
    with torch.no_grad():
        for index, (text,context) in enumerate(zip(bundle.texts,contexts_from_bundle(bundle))):
            windows,mapping,details = pair_batch_windows([text],[context],tok,device,
                cfg[role+'_max_length'],cfg[role+'_overlap'],cfg)
            # Satu jawaban diproses bertahap agar jumlah window tidak meledakkan memori.
            cls_parts=[]
            for start in range(0,len(mapping),cfg['bge_batch']):
                part={k:v[start:start+cfg['bge_batch']] for k,v in windows.items()}
                cls_parts.append(model(**part).last_hidden_state[:,0,:].float())
            cls=torch.cat(cls_parts,dim=0)
            value=mean_by_essay(cls,mapping,1)
            if role=='bge':value=torch.nn.functional.normalize(value,dim=1)
            vectors.append(value.cpu().numpy())
            window_vectors.append(cls.cpu().numpy())
            row_id=bundle.rows[index].get('row_id',f'inference_{index}')
            ids.extend(f'{row_id}_window_{j}' for j in range(len(cls)))
            coverage.append({k:v for k,v in details[0].items() if k not in ('answer_positions','reference_positions','segments')})
            if (index+1)%25==0 or index+1==len(bundle.rows):
                print(f'{role}: tiga input {index+1}/{len(bundle.rows)} jawaban selesai.',flush=True)
    values=np.concatenate(vectors).astype(np.float32)
    if role=='bge' and any(p.requires_grad for p in model.parameters()):
        raise ValueError('Encoder BGE harus frozen.')
    del model,tok
    release()
    return values,np.concatenate(window_vectors),ids,coverage


### `bge_features`

Pakai cache bila teks dan konfigurasi tokenizer sama; vektor berdimensi 1024.

In [ ]:
def bge_features(bundle, cfg, output):
    """Cache berubah jika soal, referensi, jawaban, atau resep token berubah."""
    contexts=contexts_from_bundle(bundle)
    contract={'model':cfg['bge'], 'texts':bundle.texts, 'contexts':contexts,
              'maximum':cfg['bge_max_length'],'overlap':cfg['bge_overlap'],
              'context_max_tokens':cfg['context_max_tokens'],'question_max_tokens':cfg['question_max_tokens'],
              'policy':'three_inputs_mean_CLS_then_L2_v2', 'input_format':cfg['input_format']}
    key=hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:20]
    cache=Path(cfg['output'])/'cache'/f'bge_three_inputs_{key}.npz'
    metadata=cache.with_suffix('.json')
    if cache.exists() and metadata.exists():
        proof=json.loads(metadata.read_text())
        if proof['file_sha256']!=hashlib.sha256(cache.read_bytes()).hexdigest():
            raise ValueError('Cache BGE berubah; tidak dipakai.')
        with np.load(cache,allow_pickle=False) as data:values=data['features']
        reused=True
    else:
        values,_,_,coverage=paired_pretrained_values(bundle,cfg,'bge')
        cache.parent.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(cache,features=values)
        proof={'model':cfg['bge'],'shape':list(values.shape),'frozen':True,
               'pooling':'mean paired-window CLS then L2','max_length':cfg['bge_max_length'],
               'feature_source':'Pertanyaan + Kunci_Jawaban + Jawaban', 'input_format':cfg['input_format'],
               'answer_tokens_uncovered':sum(s['answer_tokens_uncovered'] for s in coverage),
               'context_truncated_rows':sum(s['question_truncated'] or s['reference_truncated'] for s in coverage),
               'file_sha256':hashlib.sha256(cache.read_bytes()).hexdigest()}
        dump(metadata,proof)
        reused=False
    if values.shape!=(len(bundle.rows),1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4):
        raise ValueError('Fitur BGE shape/normalisasi tidak sah.')
    dump(Path(output)/'bge_encoding.json',{**proof,'reused':reused})
    return torch.from_numpy(values)


### Hitung fitur BGE untuk seluruh jawaban

Cell ini menampilkan progres encoding dan ukuran matriks vektor yang akan dipakai pada CV dan refit.

In [ ]:
bge = bge_features(bundle, cfg, output)
coverage = json.loads((output / 'bge_encoding.json').read_text())
display(pd.DataFrame([{
    'encoder_frozen': cfg['bge']['checkpoint'],
    'jumlah_vektor': bge.shape[0], 'dimensi': bge.shape[1],
    'cache_dipakai_ulang': coverage['reused'],
    'token_jawaban_tidak_tercakup': coverage['answer_tokens_uncovered'],
}]))


## 12 · Learnable Code — arsitektur S3 E4

RoBERTa → Scalar Mix 4 + gamma → BiGRU dan co-attention dua arah → masked mean. Vektor RoBERTa 768 digabung dengan proyeksi BGE 256. MLP bersama menghasilkan empat skor melalui sigmoid.

### `answer_token_mask`

Pilih posisi token jawaban yang valid.

In [ ]:
def answer_token_mask(windows, mapping, details, padding_side='right'):
    """Mask posisi respons saja; konteks/prefix/special/padding tidak dipool."""
    valid = windows['attention_mask'].bool()
    mask = torch.zeros_like(valid)
    flat = [(essay, positions) for essay, info in enumerate(details)
            for positions in info['answer_positions']]
    if len(flat) != len(valid) or padding_side not in ('left', 'right'):
        raise ValueError('Pemetaan window/posisi jawaban tidak cocok.')
    for window, (essay, positions) in enumerate(flat):
        if int(mapping[window]) != essay or not positions:
            raise ValueError('Window harus berisi token jawaban dari esai yang benar.')
        offset = valid.shape[1] - int(valid[window].sum()) if padding_side == 'left' else 0
        indices = torch.tensor(positions, dtype=torch.long, device=valid.device) + offset
        if int(indices.min()) < 0 or int(indices.max()) >= valid.shape[1]:
            raise ValueError('Posisi token jawaban keluar dari window.')
        if not bool(valid[window, indices].all()):
            raise ValueError('Token padding tidak boleh menjadi token jawaban.')
        mask[window, indices] = True
    return mask


### `check_answer_pool_inputs`

Periksa bentuk tensor dan keberadaan token valid sebelum pooling.

In [ ]:
def check_answer_pool_inputs(hidden, mask):
    if (hidden.ndim != 3 or mask is None or mask.dtype != torch.bool
            or tuple(mask.shape) != tuple(hidden.shape[:2])
            or mask.device != hidden.device):
        raise ValueError('Pooling memerlukan hidden [window,token,768] dan mask jawaban boolean.')
    if not bool(mask.any(dim=1).all()):
        raise ValueError('Setiap window harus memiliki minimal satu token jawaban.')


### `AnswerBiGRUAttention`

Kelas dasar menyediakan BiGRU dan urutan inisialisasi yang dipakai E43.

In [ ]:
class AnswerBiGRUAttention(nn.Module):
    """E26: urutan token -> BiGRU -> attention atas semua output -> proyeksi768."""
    def __init__(self, hidden_size=64, attention_hidden_size=64):
        super().__init__()
        self.gru = nn.GRU(768, hidden_size, batch_first=True, bidirectional=True)
        self.attention_hidden = nn.Linear(2*hidden_size, attention_hidden_size)
        self.attention_score = nn.Linear(attention_hidden_size, 1, bias=False)
        self.output = nn.Linear(2*hidden_size, 768)
        self.last_weights = None
        self.last_summary = None

    def forward(self, hidden, mask):
        check_answer_pool_inputs(hidden, mask)
        sequences = [row[keep] for row, keep in zip(hidden, mask)]
        lengths = torch.tensor([len(row) for row in sequences], dtype=torch.long)
        padded = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
        packed = torch.nn.utils.rnn.pack_padded_sequence(padded, lengths, batch_first=True, enforce_sorted=False)
        packed_outputs, _ = self.gru(packed)
        outputs, _ = torch.nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        valid = torch.arange(outputs.shape[1], device=outputs.device)[None] < lengths.to(outputs.device)[:,None]
        scores = self.attention_score(torch.tanh(self.attention_hidden(outputs))).squeeze(-1).float()
        weights = torch.softmax(scores.masked_fill(~valid, -torch.inf), dim=1)
        summary = (weights.unsqueeze(-1)*outputs.float()).sum(1).to(outputs.dtype)
        # Pemetaan balik bobot ke posisi token asli, supaya Excel tidak menggeser konteks/padding.
        self.last_weights = torch.zeros_like(mask, dtype=torch.float32).masked_scatter(mask, weights[valid]).detach()
        self.last_summary = summary.detach()
        return self.output(summary)


### `reference_token_mask`

Pilih posisi token referensi untuk jalur pencocokan.

In [ ]:
def reference_token_mask(windows, mapping, details, padding_side='right'):
    """Pemetaan posisi referensi memakai offset padding yang sama dengan respons."""
    converted = [{**info, 'answer_positions': info['reference_positions']} for info in details]
    return answer_token_mask(windows, mapping, converted, padding_side)


### `AnswerBiGRUCoAttentionMeanPool`

Cocokkan token jawaban dan referensi, lalu rata-ratakan fitur token jawaban valid. Bobot matching co-attention bergantung pada representasi token.

In [ ]:
class AnswerBiGRUCoAttentionMeanPool(AnswerBiGRUAttention):
    """E43: BiGRU + co-attention respons–referensi, lalu masked mean token matching128."""
    def __init__(self, hidden_size=64, attention_hidden_size=64):
        super().__init__(hidden_size, attention_hidden_size)
        dim = 2 * hidden_size
        # Sesudah layer acuan: inisialisasi BiGRU/attention/output E26 tetap sama.
        self.reference_projection = nn.Linear(768, dim)
        self.matching_projection = nn.Linear(5 * dim, dim)
        self.capture_alignment = False
        self.last_cross_attention = None
        self.last_reference_weights = None
        self.last_weights = None
        self.last_response_coattention_weights = None
        # Semua bobot bersama dibuat dengan urutan RNG E42 yang sama.
        # Dua layer skor pooling acuan kemudian dihapus; tidak ada parameter attention pooling yang dilatih.
        del self.attention_hidden
        del self.attention_score

    def forward(self, hidden, mask, reference_mask):
        check_answer_pool_inputs(hidden, mask)
        check_answer_pool_inputs(hidden, reference_mask)
        if bool((mask & reference_mask).any()):
            raise ValueError('Mask respons dan referensi harus terpisah.')
        responses = [row[keep] for row, keep in zip(hidden, mask)]
        references = [row[keep] for row, keep in zip(hidden, reference_mask)]
        lengths = torch.tensor([len(row) for row in responses], dtype=torch.long)
        ref_lengths = torch.tensor([len(row) for row in references], dtype=torch.long)
        padded = torch.nn.utils.rnn.pad_sequence(responses, batch_first=True)
        packed = torch.nn.utils.rnn.pack_padded_sequence(padded, lengths, batch_first=True, enforce_sorted=False)
        packed_outputs, _ = self.gru(packed)
        response, _ = torch.nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        reference = self.reference_projection(torch.nn.utils.rnn.pad_sequence(references, batch_first=True))
        valid = torch.arange(response.shape[1], device=hidden.device)[None] < lengths.to(hidden.device)[:, None]
        ref_valid = torch.arange(reference.shape[1], device=hidden.device)[None] < ref_lengths.to(hidden.device)[:, None]
        # Softmax/produk skor FP32 juga saat encoder memakai AMP. Kedua urutan punya >=1 token.
        with torch.autocast(device_type=hidden.device.type, enabled=False):
            response32, reference32 = response.float(), reference.float()
            scores = torch.bmm(response32, reference32.transpose(1, 2)) / math.sqrt(response.shape[-1])
            a_to_r = torch.softmax(scores.masked_fill(~ref_valid[:, None, :], -torch.inf), dim=-1)
            a_to_r = a_to_r.masked_fill(~valid[:, :, None], 0.)
            r_to_a = torch.softmax(scores.transpose(1, 2).masked_fill(~valid[:, None, :], -torch.inf), dim=-1)
            r_to_a = r_to_a.masked_fill(~ref_valid[:, :, None], 0.)
            aligned_reference = torch.bmm(a_to_r, reference32)
            # Jalur balik respons→referensi→respons; bukan attention atas satu vektor BGE.
            round_trip = torch.bmm(a_to_r, torch.bmm(r_to_a, response32))
            matching = torch.cat([response32, aligned_reference, response32 - aligned_reference,
                                  response32 * aligned_reference, round_trip], dim=-1)
        matched = torch.tanh(self.matching_projection(matching.to(response.dtype)))
        # Setiap token matching yang valid mendapat bobot 1/N.
        # Padding berbobot 0; mean ini merangkum token dalam satu window.
        pooling_weights = valid.float() / lengths.to(hidden.device).unsqueeze(1)
        summary = (pooling_weights.unsqueeze(-1) * matched.float()).sum(1).to(response.dtype)
        self.last_weights = torch.zeros_like(mask, dtype=torch.float32).masked_scatter(mask, pooling_weights[valid]).detach()
        # Rata-rata bobot arah referensi→respons untuk inspeksi, terpisah dari pooling.
        response_importance = r_to_a.sum(1) / ref_lengths.to(hidden.device)[:, None]
        self.last_response_coattention_weights = torch.zeros_like(mask, dtype=torch.float32).masked_scatter(
            mask, response_importance[valid]).detach()
        self.last_summary = summary.detach()
        reference_importance = a_to_r.sum(1) / lengths.to(hidden.device)[:, None]
        self.last_reference_weights = torch.zeros_like(reference_mask, dtype=torch.float32).masked_scatter(
            reference_mask, reference_importance[ref_valid]).detach()
        # Matriks hanya ditahan untuk ekspor inspeksi, bukan graph training atau seluruh dataset.
        self.last_cross_attention = a_to_r.detach() if self.capture_alignment else None
        return self.output(summary)


### `ScalarMix` dan `MultiTraitModel`

Scalar Mix mempelajari bobot empat layer terakhir dan gamma. MLP mengubah fitur gabungan 1024 menjadi empat skor normalisasi melalui sigmoid.

In [ ]:
class ScalarMix(nn.Module):
    """Campurkan k layer Transformer terakhir dengan bobot global yang dipelajari."""
    def __init__(self, number_of_layers=4, gamma_initial=1.0):
        super().__init__()
        if type(number_of_layers) is not int or number_of_layers < 1:
            raise ValueError('Jumlah layer Scalar Mix harus bilangan bulat positif.')
        self.number_of_layers = number_of_layers
        # Empat logit nol memberi bobot0.25; gamma awal1 mempertahankan keluaran awal E41.
        self.logits = nn.Parameter(torch.zeros(number_of_layers))
        self.gamma = nn.Parameter(torch.full((1,), float(gamma_initial)))
        self.capture_stats = False

    def weights(self):
        return torch.softmax(self.logits, dim=0)

    def forward(self, hidden_states, attention_mask=None):
        # hidden_states[0] adalah embeddings; hanya keluaran layer Transformer dipilih.
        if hidden_states is None or len(hidden_states)-1 < self.number_of_layers:
            raise ValueError('Encoder tidak menyediakan cukup layer Transformer untuk Scalar Mix.')
        selected = hidden_states[-self.number_of_layers:]
        if any(value.shape != selected[0].shape for value in selected):
            raise ValueError('Semua tensor layer Scalar Mix harus mempunyai shape yang sama.')
        weights = self.weights().to(dtype=selected[0].dtype)
        # Penjumlahan bertahap menghindari stack tambahan [4,batch,token,768].
        mixed = torch.zeros_like(selected[0])
        for weight, value in zip(weights, selected):
            mixed = mixed + weight * value
        scaled = self.gamma.to(dtype=mixed.dtype) * mixed
        # Statistik hanya saat inspeksi fitur, tanpa graph gradien atau sinkronisasi CPU tiap train step.
        if self.capture_stats:
            with torch.no_grad():
                valid = attention_mask.bool() if attention_mask is not None else torch.ones_like(mixed[..., 0], dtype=torch.bool)
                counts = valid.sum(1).clamp_min(1)
                def average_norm(value):
                    return ((value.float().norm(dim=-1)*valid).sum(1)/counts).unsqueeze(1)
                self.norm_before = average_norm(mixed)
                self.norm_after = average_norm(scaled)
                self.norm_last_layer = average_norm(hidden_states[-1])
        return scaled

    def report(self, encoder_layers):
        weights = self.weights().detach().float().cpu().tolist()
        logits = self.logits.detach().float().cpu().tolist()
        start = encoder_layers-self.number_of_layers
        return [{'Indeks Layer RoBERTa': start+i, 'Layer ke-': start+i+1,
                 'Logit': logits[i], 'Bobot Softmax': weights[i],
                 'Bobot Awal': 1/self.number_of_layers, 'Gamma': float(self.gamma.detach().cpu().item())}
                for i in range(self.number_of_layers)]


class MultiTraitModel(nn.Module):
    """S3 saja: representasi dua encoder digabung dan dinilai oleh MLP."""
    def __init__(self, scenario, roberta=None, dropout=.1, sequence_hidden=64,
                 s3_mlp_dropout=.1, pooling_attention_hidden=64, scalar_mix_layers=4, gamma_initial=1.0):
        super().__init__()
        if scenario != 'S3' or roberta is None:
            raise ValueError('E43 memerlukan S3 dan encoder RoBERTa; tidak membuat skenario lain.')
        self.scenario, self.roberta = scenario, roberta
        roberta.requires_grad_(True)
        self.dropout = nn.Dropout(dropout)  # Atribut historis; forward S3 memakai dropout di head.
        # Urutan inisialisasi layer yang tidak berubah tetap mengikuti E30.
        self.roberta_norm = nn.LayerNorm(768)
        self.projection = nn.Sequential(nn.Linear(1024, 256), nn.LayerNorm(256))
        self.head = nn.Sequential(nn.Linear(1024, 256), nn.GELU(),
                                  nn.Dropout(s3_mlp_dropout), nn.Linear(256, len(TRAITS)), nn.Sigmoid())
        self.sequence_pool = AnswerBiGRUCoAttentionMeanPool(sequence_hidden, pooling_attention_hidden)
        if roberta.config.num_hidden_layers < scalar_mix_layers:
            raise ValueError('RoBERTa mempunyai lebih sedikit layer daripada scalar_mix_layers.')
        # Inisialisasi nol tidak mengubah RNG layer E30 yang sudah dibentuk.
        self.scalar_mix = ScalarMix(scalar_mix_layers, gamma_initial)

    def forward(self, windows=None, mapping=None, n_essays=None, bge=None,
                return_features=False, answer_mask=None, reference_mask=None):
        encoded = self.roberta(**windows, output_hidden_states=True, return_dict=True)
        self.scalar_mix.capture_stats = return_features
        hidden = self.scalar_mix(encoded.hidden_states, windows['attention_mask'])
        cls = encoded.last_hidden_state[:, 0, :]  # CLS referensi inspeksi tetap layer terakhir.
        # BiGRU64/arah menghasilkan128fitur per token; pooling diproyeksikan ke768.
        pooled = self.sequence_pool(hidden, answer_mask, reference_mask)
        essay = mean_by_essay(pooled, mapping, n_essays)
        features = {}
        if return_features:
            for name, value in [('mix_norm_before', self.scalar_mix.norm_before),
                                ('mix_norm_after', self.scalar_mix.norm_after),
                                ('mix_norm_last', self.scalar_mix.norm_last_layer)]:
                features[name] = mean_by_essay(value, mapping, n_essays)
        features['roberta_cls_reference'] = mean_by_essay(cls, mapping, n_essays)
        features['answer_pool768'] = essay
        features['bigru_summary'] = mean_by_essay(self.sequence_pool.last_summary, mapping, n_essays)
        features.update(roberta=essay, roberta_window_cls=cls)
        if bge is None or bge.shape != (n_essays, 1024):
            raise ValueError('BGE frozen memerlukan satu vektor1024 untuk setiap jawaban.')
        features['bge'] = bge
        features['roberta_layernorm'] = self.roberta_norm(essay)
        features['projection'] = self.projection(bge)
        features['concat'] = torch.cat([features['roberta_layernorm'], features['projection']], dim=1)
        prediction = self.head(features['concat'])
        if prediction.shape != (n_essays, len(TRAITS)):
            raise ValueError('S3 harus menghasilkan empat skor per jawaban.')
        return (prediction, features) if return_features else prediction


### `Engine`

Siapkan input batch dan kembalikan prediksi dari skala normalisasi ke poin tiap trait.

In [ ]:
class Engine:
    """Jalankan tiga teks melalui model fusion S3 yang sama."""
    def __init__(self, model, texts, bge, cfg, device, roberta_tokenizer=None, contexts=None):
        if model.scenario != 'S3':
            raise ValueError('Engine E43 hanya menjalankan S3.')
        self.model, self.texts, self.bge, self.cfg, self.device = model, texts, bge, cfg, device
        self.contexts = contexts
        if contexts is None or len(contexts) != len(texts):
            raise ValueError('Setiap jawaban memerlukan satu pasangan soal dan referensi.')
        self.tokenizer = roberta_tokenizer or tokenizer(cfg, 'roberta')

    def forward(self, indices, return_features=False):
        ids = [int(i) for i in indices]
        features = self.bge[ids].to(self.device)
        windows, mapping, details = pair_batch_windows(
            [self.texts[i] for i in ids], [self.contexts[i] for i in ids],
            self.tokenizer, self.device, self.cfg['roberta_max_length'],
            self.cfg['roberta_overlap'], self.cfg)
        mask = answer_token_mask(windows, mapping, details, self.tokenizer.padding_side)
        ref_mask = reference_token_mask(windows, mapping, details, self.tokenizer.padding_side)
        self.model.sequence_pool.capture_alignment = return_features
        if return_features:
            self.pooling_batch = (windows, mask)
            self.coattention_batch = (windows, mask, ref_mask)
        return self.model(windows=windows, mapping=mapping, n_essays=len(ids), bge=features,
                          return_features=return_features, answer_mask=mask, reference_mask=ref_mask)

    def predict(self, indices):
        # Mode prediksi: nonaktifkan dropout. Pembacaan di bawah memakai no_grad.
        self.model.eval()
        values = []
        # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
        with torch.no_grad():
            for start in range(0, len(indices), self.cfg['micro_batch']):
                ids = indices[start:start+self.cfg['micro_batch']]
                values.append(self.forward(ids).float().cpu().numpy())
        scaled = np.concatenate(values)
        # Kembalikan setiap keluaran ke satuan poin trait.
        return scaled * SCORE_SPAN + SCORE_MIN


## 13 · Inisialisasi model dan spesifikasi

### `seed_everything`

Atur pengacakan training pada awal setiap fit mengikuti seed E43.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### `make_model`

Buat model baru untuk setiap fold dan refit.

In [ ]:
def make_model(scenario, cfg):
    if scenario != 'S3':
        raise ValueError('E43 hanya membuat model S3.')
    return MultiTraitModel('S3', encoder(cfg, 'roberta'), dropout=cfg['dropout'],
                            sequence_hidden=cfg['s3_sequence_hidden'],
                            s3_mlp_dropout=cfg['s3_mlp_dropout'],
                            pooling_attention_hidden=cfg['pooling_attention_hidden'],
                            scalar_mix_layers=cfg['scalar_mix_layers'], gamma_initial=cfg['scalar_mix_gamma_init'])


In [ ]:
display(pd.DataFrame([
    ('RoBERTa', '768 per token', cfg['s3_encoder_lr']),
    ('Scalar Mix 4 + gamma', '768 per token', cfg['scalar_mix_lr']),
    ('BiGRU + co-attention', 2 * cfg['s3_sequence_hidden'], cfg['s3_layers_lr']),
    ('Masked mean + proyeksi RoBERTa', 768, cfg['s3_layers_lr']),
    ('BGE-M3 frozen', 1024, None),
    ('Proyeksi BGE', 256, cfg['projection_lr']),
    ('Concat → MLP', f'1024 → 256 → {len(TRAITS)} → sigmoid', cfg['s3_mlp_lr']),
], columns=['bagian', 'dimensi', 'learning_rate']))
print('Urutan empat keluaran MLP:', ', '.join(name for name, *_ in TRAITS))


## 14 · Metrik evaluasi

RMSE dan Pearson memakai prediksi mentah. QWK dan Gwet AC2 memakai kategori terdekat.

In [ ]:
import warnings
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score

def ordinal_prediction(values, trait):
    cats=np.asarray(trait['categories'],dtype=float)
    raw=np.asarray(values,dtype=float)
    if not np.isfinite(raw).all():raise ValueError('Prediksi nonfinite')
    a=np.clip(raw,trait['min'],trait['max'])
    # argmin memilih kategori pertama (lebih rendah) ketika jarak sama.
    return cats[np.abs(a[...,None]-cats).argmin(-1)].astype(int)

def gwet_ac2(teacher, predicted, categories):
    a=np.asarray(teacher);b=np.asarray(predicted);c=np.asarray(categories,dtype=float)
    if len(a)!=len(b) or len(a)<2: raise ValueError('AC2 perlu minimal dua pasangan')
    if not np.isin(a,c).all() or not np.isin(b,c).all(): raise ValueError('Kategori AC2 tidak sah')
    q=len(c); weights=1-((c[:,None]-c[None,:])/(c.max()-c.min()))**2
    ai=np.searchsorted(c,a); bi=np.searchsorted(c,b)
    pa=weights[ai,bi].mean()
    pi=(np.bincount(ai,minlength=q)+np.bincount(bi,minlength=q))/(2*len(a))
    pe=weights.sum()*np.sum(pi*(1-pi))/(q*(q-1))
    return float((pa-pe)/(1-pe))

def evaluate(teacher, predicted, rubric):
    y=np.asarray(teacher,dtype=float);p=np.asarray(predicted,dtype=float)
    if y.shape!=p.shape or y.ndim!=2 or y.shape[1]!=4: raise ValueError('Metrik memerlukan [N,4] yang sama')
    if not np.isfinite(y).all() or not np.isfinite(p).all(): raise ValueError('Label/prediksi nonfinite')
    result={}
    for j,t in enumerate(rubric['traits']):
        a,b=y[:,j],p[:,j]; cats=t['categories']
        if not np.isin(a,cats).all(): raise ValueError('Label harus kategori guru yang sah')
        o=ordinal_prediction(b,t); reasons={}
        rmse=float(np.sqrt(np.mean((a-b)**2))) if len(a) else None
        pearson=None
        if len(a)>=2 and np.ptp(a)>0 and np.ptp(b)>0: pearson=float(pearsonr(a,b).statistic)
        else: reasons['Pearson']='Minimal dua pasangan dan kedua input tidak konstan.'
        qwk=None;ac2=None
        if len(a)>=2:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore'); v=cohen_kappa_score(a,o,weights='quadratic',labels=cats)
            if np.isfinite(v): qwk=float(v)
            else: reasons['QWK']='Expected disagreement nol; koefisien undefined.'
            ac2=gwet_ac2(a,o,cats)
        else: reasons.update({'QWK':'Minimal dua pasangan.','AC2':'Minimal dua pasangan.'})
        result[t['name']]={'RMSE':rmse,'Pearson':pearson,'QWK':qwk,'AC2':ac2,
             'out_of_range_fraction':float(((b<t['min']) | (b>t['max'])).mean()) if len(b) else None,
             'undefined_reasons':reasons}
    return result


### `metrics` dan `predictions_array`

Periksa bentuk empat skor, lalu hitung metrik per trait. Pemilihan epoch memakai RMSE dari seluruh skor validasi pada skala asli.

In [ ]:
def metrics(teacher, prediction):
    """RMSE validasi pada seluruh pasangan skor asli, serta metrik per trait."""
    per_trait = evaluate(teacher, prediction, RUBRIC)
    residual = np.asarray(teacher, dtype=float) - np.asarray(prediction, dtype=float)
    return {'RMSE': float(np.sqrt(np.mean(residual ** 2))), 'traits': per_trait}


def predictions_array(values, indices):
    array = np.asarray(values, dtype=float)
    if array.shape != (len(indices), len(TRAITS)) or not np.isfinite(array).all():
        raise ValueError('Prediksi harus finite dan berukuran [jumlah jawaban, 4].')
    return array


## 15 · Learnable Code — optimizer, loss, dan satu fit

### `optimizer_groups`

Kelompokkan parameter dengan LR RoBERTa 5e-6 dan LR komponen lain 1e-4.

In [ ]:
def optimizer_groups(model, cfg):
    """Lima kelompok optimizer S3; semua parameter trainable tercakup tepat sekali."""
    buckets = {'roberta': [], 'projection': [], 'head': [], 'pooling_norm': [], 'scalar_mix': []}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith('roberta.'):
            group = 'roberta'
        elif name.startswith('scalar_mix.'):
            group = 'scalar_mix'
        elif name.startswith('projection.'):
            group = 'projection'
        elif name.startswith('head.'):
            group = 'head'
        else:
            group = 'pooling_norm'
        buckets[group].append(parameter)
    rates = {'roberta': cfg['s3_encoder_lr'],
             'projection': cfg['projection_lr'],
             'head': cfg['s3_mlp_lr'],
             'pooling_norm': cfg['s3_layers_lr'], 'scalar_mix': cfg['scalar_mix_lr']}
    result = [{'name': name, 'params': params, 'lr': rates[name]} for name, params in buckets.items() if params]
    ids = [id(p) for group in result for p in group['params']]
    expected = {id(p) for p in model.parameters() if p.requires_grad}
    if len(ids) != len(set(ids)) or set(ids) != expected:
        raise ValueError('Optimizer harus mencakup setiap parameter trainable tepat sekali.')
    return result


### `fit`

Latih satu model dengan AdamW dan MSE dari empat target skala 0–1. Saat CV, pilih epoch dengan RMSE validasi terkecil.

In [ ]:
def fit(scenario, training, validation, bundle, bge, cfg, epochs, seed):
    seed_everything(seed)
    device = device_name(cfg)
    # Buat model baru untuk fold ini, lalu tempatkan di GPU atau CPU yang tersedia.
    model = make_model(scenario, cfg).to(device)
    engine = Engine(model, bundle.texts, bge, cfg, device, contexts=contexts_from_bundle(bundle))
    groups = optimizer_groups(model, cfg)
    learning_rates = {group['name']: group['lr'] for group in groups}
    # AdamW memperbarui kelompok parameter sesuai learning rate masing-masing.
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader([int(i) for i in training], batch_size=cfg['effective_batch'],
                        shuffle=True, generator=generator, num_workers=0)
    total_steps = len(loader)*epochs
    warmup = min(int(cfg['warmup_ratio']*total_steps), max(0, total_steps-1))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)
    use_cuda = device == 'cuda'
    dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda and dtype == torch.float16)
    selected_parameters = {}
    for name, parameter in model.named_parameters():
        group = name.split('.')[0]
        if parameter.requires_grad and group not in selected_parameters:
            selected_parameters[group] = (name, parameter.detach().reshape(-1)[:32].cpu().clone())
    best_state, best_epoch, best_rmse = None, 0, float('inf')
    patience = cfg['early_stopping_patience']
    without_improvement, early_stopped = 0, False
    history, active_steps, optimizer_steps, skipped_steps = [], 0, 0, 0
    for epoch in range(1, epochs+1):
        model.train()
        loss_total = 0.
        for effective in loader:
            batch = effective.tolist()
            optimizer.zero_grad(set_to_none=True)
            for start in range(0, len(batch), cfg['micro_batch']):
                ids = batch[start:start+cfg['micro_batch']]
                # Normalisasi per trait memberi bobot sama pada empat dimensi loss.
                targets = torch.as_tensor((bundle.y[ids] - SCORE_MIN) / SCORE_SPAN,
                                          dtype=torch.float32, device=device)
                with torch.autocast(device_type=device, dtype=dtype, enabled=use_cuda):
                    prediction = engine.forward(ids).float()
                    if prediction.shape != targets.shape:
                        raise ValueError('Shape prediksi/target harus [B,4], tanpa broadcasting.')
                    # MSE mengukur selisih kuadrat antara prediksi dan target; optimizer berusaha memperkecilnya.
                    loss = torch.mean((prediction-targets)**2)
                if not torch.isfinite(loss):
                    raise RuntimeError('Loss nonfinite; run dihentikan, tidak diklaim berhasil.')
                # Hitung gradien dan bobot microbatch sesuai ukuran batch efektif aktual.
                scaler.scale(loss*(len(ids)/len(batch))).backward()
                loss_total += loss.item()*len(ids)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
            lr_active = any(g['lr'] > 0 for g in optimizer.param_groups)
            previous_scale = scaler.get_scale()
            # Perbarui bobot; GradScaler bisa melewati langkah yang overflow pada AMP.
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.is_enabled() and scaler.get_scale() < previous_scale
            if skipped:
                skipped_steps += 1
            else:
                active_steps += int(lr_active)
                scheduler.step()
                optimizer_steps += 1
        row = {'epoch': epoch, 'train_mse_scaled': loss_total/len(training),
               'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
               'effective_batch_target': cfg['effective_batch'], 'micro_batch': cfg['micro_batch']}
        if validation is not None:
            predicted = engine.predict(validation)
            row['validation'] = metrics(bundle.y[validation], predicted)
            # Simpan epoch dengan RMSE validasi terkecil; data uji akhir tidak ikut memilih epoch.
            if row['validation']['RMSE'] < best_rmse:
                best_rmse, best_epoch = row['validation']['RMSE'], epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                without_improvement = 0
            else:
                without_improvement += 1
            row.update(best_epoch_so_far=best_epoch, epochs_without_improvement=without_improvement)
            message = (f" | RMSE validasi: {row['validation']['RMSE']:.4f} poin"
                       f" | terbaik: epoch {best_epoch}, {best_rmse:.4f} poin"
                       f" | tanpa perbaikan: {without_improvement}/{patience}")
        else:
            best_epoch = epoch
            message = ' | pelatihan final; data uji belum dibuka'
        history.append(row)
        print(f'{scenario} · epoch {epoch}/{epochs} · loss latih (MSE, target 0–1): {row["train_mse_scaled"]:.6f}'+message, flush=True)
        # Hanya fold validasi yang boleh menghentikan latihan; data uji tidak dibaca.
        if validation is not None and without_improvement >= patience and epoch < epochs:
            early_stopped = True
        if early_stopped:
            print(f'{scenario}: early stopping setelah {epoch} epoch; RMSE validasi tidak membaik selama {patience} epoch. '
                  f'Bobot epoch {best_epoch} dipulihkan.', flush=True)
            break
    if active_steps == 0:
        raise RuntimeError('Tidak ada pembaruan dengan LR aktif.')
    named = dict(model.named_parameters())
    changed = {group: not torch.equal(before, named[name].detach().reshape(-1)[:32].cpu())
               for group, (name, before) in selected_parameters.items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    info = {'scenario': scenario, 'best_epoch': best_epoch, 'epochs_completed': len(history),
            'max_epochs': epochs, 'early_stopping_patience': patience if validation is not None else None,
            'early_stopped': early_stopped, 'optimizer_learning_rates': learning_rates,
            'best_validation_rmse': best_rmse if validation is not None else None,
            'stop_reason': ('RMSE validasi tidak membaik selama batas kesabaran' if early_stopped else
                            'Batas maksimum epoch tercapai' if validation is not None else
                            'Jumlah epoch final yang dipilih dari CV selesai'),
            'history': history, 'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
            'parameter_samples_changed': changed, 'BGE_encoder_trained': False,
            'selection': 'minimum validation RMSE in original score units' if validation is not None else 'fixed refit epochs selected from CV'}
    # Bobot diambil SESUDAH best_state dipulihkan; sesuai model yang dievaluasi.
    info['scalar_mix_weights'] = model.scalar_mix.report(model.roberta.config.num_hidden_layers)
    info['scalar_mix_layers'] = cfg['scalar_mix_layers']
    print('Bobot Scalar Mix pada checkpoint terpilih:', flush=True)
    print(pd.DataFrame(info['scalar_mix_weights']).to_string(index=False), flush=True)
    return model, engine, info


## 16 · Validasi silang pada development

### `run_cv`

Latih lima model E4 pada lima fold. Ambil median epoch terbaik lalu bulatkan ke atas untuk menentukan durasi refit.

In [ ]:
def run_cv(bundle, manifest, bge, cfg, output):
    """Validasi silang satu resep E4 pada development."""
    results = []
    for fold in manifest['folds']:
        number, validation = fold['fold'], fold['validation']
        path = output / f'fold_{number}.json'
        identity = {'split': manifest['split_sha256'], 'fold': number, 'validation': validation}
        if path.exists():
            result = json.loads(path.read_text())
            if result['identity'] != identity:
                raise ValueError(f'Cache fold memakai split berbeda: {path}')
            print(f'E4 · fold {number}: hasil selesai dipakai ulang.', flush=True)
        else:
            print(f'E4 · fold {number}/{cfg["folds"]}', flush=True)
            model, engine, info = fit('S3', fold['train'], validation, bundle, bge,
                                      cfg, cfg['epochs'], cfg['seed'] + number)
            prediction = predictions_array(engine.predict(validation), validation)
            result = {'identity': identity, 'prediction': prediction.tolist(), 'training': info}
            dump(path, result)
            del model, engine
            release()
        prediction = predictions_array(result['prediction'], validation)
        evaluation = metrics(bundle.y[validation], prediction)
        if not math.isclose(evaluation['RMSE'],
                            result['training']['best_validation_rmse'], abs_tol=1e-6):
            raise ValueError(f'Prediksi dan epoch terpilih tidak cocok: {path}')
        results.append({**result, 'metrics': evaluation})
    indices = [i for result in results for i in result['identity']['validation']]
    if sorted(indices) != sorted(manifest['development']):
        raise ValueError('Setiap jawaban development harus memiliki tepat satu prediksi OOF.')
    return {
        'eksperimen': 'E4', 'mean_cv_RMSE': float(np.mean([
            r['metrics']['RMSE'] for r in results
        ])),
        'epoch_refit': math.ceil(np.median([r['training']['best_epoch'] for r in results])),
    }, results


### Jalankan lima fold

Output berisi progres per fold, epoch terbaik tiap fold, dan ringkasan CV.

In [ ]:
cv_summary, folds = run_cv(bundle, manifest, bge, cfg, output)
display(pd.DataFrame([{
    'fold': result['identity']['fold'],
    'epoch_terpilih': result['training']['best_epoch'],
    'RMSE_validasi': result['metrics']['RMSE'],
} for result in folds]))
display(pd.DataFrame([cv_summary]))


## 17 · Refit E4 pada seluruh development

### `save_model`

Simpan bobot dan metadata dari model yang selesai dilatih.

In [ ]:
def save_model(path, model, cfg, bundle, manifest, info):
    payload = {'scope': SCOPE, 'trained': True, 'scenario': model.scenario,
               'source_sha256': bundle.source_sha256, 'split_sha256': manifest['split_sha256'],
               'config': cfg, 'training': info,
               'state': {k: v.detach().cpu() for k, v in model.state_dict().items()}}
    torch.save(payload, path)


### `refit_checkpoint`

Latih sekali pada seluruh development menggunakan jumlah epoch dari CV, lalu simpan checkpoint.

In [ ]:
def refit_checkpoint(epochs, bundle, manifest, bge, cfg, output):
    """Latih pada development dan simpan checkpoint sebelum cell test dijalankan."""
    identity = {'refit_epochs': epochs, 'split': manifest['split_sha256']}
    model_path = output / 'final.pt'
    refit_path = output / 'refit_result.json'
    test_path = output / 'test_result.json'
    # Format lama menyimpan informasi refit bersama hasil test.
    for saved in (refit_path, test_path):
        if saved.is_file():
            result = json.loads(saved.read_text())
            if (result['identity'] != identity or not model_path.is_file()
                or digest(model_path) != result['model_sha256']):
                raise ValueError('Metadata refit dan checkpoint final berbeda.')
            print('Checkpoint refit yang sudah selesai dipakai ulang.', flush=True)
            return result

    print(f'Refit E4 · {epochs} epoch pada seluruh development.', flush=True)
    model, engine, info = fit('S3', manifest['development'], None, bundle, bge, cfg,
                              epochs, cfg['seed'])
    pending = model_path.with_suffix('.pending.pt')
    save_model(pending, model, cfg, bundle, manifest, info)
    pending.replace(model_path)
    result = {'identity': identity, 'training': info, 'model_sha256': digest(model_path)}
    dump(refit_path, result)
    del model, engine
    release()
    return result


### Jalankan refit dan simpan checkpoint

Output menunjukkan bobot Scalar Mix final dan lokasi checkpoint.

In [ ]:
refit_result = refit_checkpoint(cv_summary['epoch_refit'], bundle, manifest, bge, cfg, output)
final_info = refit_result['training']
display(pd.DataFrame(final_info['scalar_mix_weights']))
print(f'Checkpoint final: {output / "final.pt"}')


## 18 · Evaluasi test

Baca checkpoint dari tahap refit. Tabel output berisi RMSE, Pearson, QWK, dan Gwet AC2 untuk setiap trait.

### `evaluate_test`

Prediksi data test dan simpan hasilnya agar bisa dibaca ulang saat cell diulang.

In [ ]:
def evaluate_test(refit_result, bundle, manifest, bge, cfg, output):
    """Baca checkpoint refit dan hitung satu prediksi untuk setiap jawaban test."""
    complete = output / 'test_result.json'
    model_path = output / 'final.pt'
    if digest(model_path) != refit_result['model_sha256']:
        raise ValueError('Checkpoint yang akan dievaluasi berbeda dari checkpoint refit.')
    test = manifest['test']
    if complete.exists():
        result = json.loads(complete.read_text())
        if (result['identity'] != refit_result['identity']
            or result['model_sha256'] != refit_result['model_sha256']):
            raise ValueError('Hasil test dan checkpoint final berbeda.')
        print('Hasil test final yang sudah selesai dipakai ulang.', flush=True)
        return predictions_array(result['prediction'], test)

    payload = torch.load(model_path, map_location='cpu', weights_only=True)
    device = device_name(cfg)
    model = make_model('S3', cfg)
    model.load_state_dict(payload['state'], strict=True)
    del payload
    model.to(device)
    engine = Engine(model, bundle.texts, bge, cfg, device, contexts=contexts_from_bundle(bundle))
    prediction = predictions_array(engine.predict(test), test)
    dump(complete, {**refit_result, 'prediction': prediction.tolist()})
    del model, engine
    release()
    return prediction


### `trait_rows`, `prediction_rows`, dan `export_results`

Bentuk tabel evaluasi dan prediksi yang digunakan di layar dan workbook.

In [ ]:
def excel_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, default=str)
    if isinstance(value, float) and not math.isfinite(value):
        return None  # Tidak tercakup/undefined: tetap kosong, bukan nol.
    if value is pd.NA:
        return None
    return value


In [ ]:
def write_excel(path, sheets, metadata):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    book = Workbook(write_only=True)
    used = set()
    counts = {}
    all_sheets = {'Info': pd.DataFrame([{'field': k, 'value': v} for k, v in metadata.items()]), **sheets}
    for name, frame in all_sheets.items():
        frame = frame if isinstance(frame, pd.DataFrame) else pd.DataFrame(frame)
        if len(frame.columns) > 16384:
            raise ValueError('Terlalu banyak kolom untuk Excel; tidak dipotong diam-diam.')
        headers = [str(x) for x in frame.columns]
        chunks = max(1, math.ceil(len(frame)/1048575))
        for part in range(chunks):
            base = re.sub(r'[\\/*?:\[\]]', '_', name)
            suffix = f'_{part+1}' if chunks > 1 else ''
            title = base[:31-len(suffix)]+suffix
            if title in used:
                raise ValueError('Nama sheet bertabrakan setelah dipendekkan: '+title)
            used.add(title)
            sheet = book.create_sheet(title)
            sheet.freeze_panes = 'B2'
            heading = []
            for text in headers:
                cell = WriteOnlyCell(sheet, value=text)
                cell.font = Font(bold=True, color='FFFFFF')
                cell.fill = PatternFill('solid', fgColor='172035')
                cell.data_type = 's'
                heading.append(cell)
            sheet.append(heading)
            subset = frame.iloc[part*1048575:(part+1)*1048575]
            for row in subset.itertuples(index=False, name=None):
                values = []
                for value in row:
                    value = excel_value(value)
                    if isinstance(value, str):
                        if len(value) > 32767:
                            raise ValueError('Teks melebihi batas satu sel Excel; ekspor dihentikan agar tidak terpotong.')
                        cell = WriteOnlyCell(sheet, value=value)
                        cell.data_type = 's'  # Teks siswa '=...' tidak dijalankan sebagai formula.
                        values.append(cell)
                    else:
                        values.append(value)
                sheet.append(values)
            counts[title] = {'rows': len(subset), 'columns': len(headers)}
    temporary = path.with_name(path.stem+'.pending.xlsx')
    book.save(temporary)
    temporary.replace(path)
    return counts


In [ ]:
def trait_rows(values, **keys):
    return [{**keys, 'trait': name,
             'RMSE': stats['RMSE'], 'Pearson': stats['Pearson'],
             'QWK': stats['QWK'], 'Gwet AC2': stats['AC2'],
             'proporsi_di_luar_skala': stats['out_of_range_fraction'],
             'alasan_metrik_kosong': stats['undefined_reasons']}
            for name, stats in values['traits'].items()]


def prediction_rows(bundle, indices, prediction, **keys):
    scores = predictions_array(prediction, indices)
    categories = np.column_stack([
        ordinal_prediction(scores[:, i], trait) for i, trait in enumerate(RUBRIC['traits'])
    ])
    rows = []
    for j, i in enumerate(indices):
        source = bundle.rows[i]
        row = {**keys, 'id_jawaban': source['row_id'], 'baris_excel': source['source_row'],
               'no_soal': source['response_number'], 'kelompok': int(bundle.groups[i])}
        for col, (name, _, _, _) in enumerate(TRAITS):
            actual, raw = bundle.y[i, col], scores[j, col]
            row.update({f'{name}_label': int(actual), f'{name}_prediksi_mentah': float(raw),
                        f'{name}_prediksi_kategori': int(categories[j, col]),
                        f'{name}_error_abs': float(abs(raw - actual))})
        rows.append(row)
    return rows


def history_rows(info, **keys):
    return [{**keys, **{k: v for k, v in row.items() if k != 'validation'},
             'validation_RMSE': row.get('validation', {}).get('RMSE')}
            for row in info['history']]


def export_results(path, bundle, manifest, cfg, cv_summary, folds, final_info, prediction):
    cv_fold, oof_rows, history, mixture, indices, predictions = [], [], [], [], [], []
    for result in folds:
        fold = result['identity']['fold']
        valid = result['identity']['validation']
        info = result['training']
        cv_fold.extend(trait_rows(result['metrics'], fold=fold,
                                  epoch_terpilih=info['best_epoch'], n=len(valid)))
        oof_rows.extend(prediction_rows(bundle, valid, result['prediction'], fold=fold))
        history.extend(history_rows(info, fold=fold))
        mixture.extend({'fold': fold, **r} for r in info['scalar_mix_weights'])
        indices.extend(valid)
        predictions.extend(result['prediction'])
    cv_oof = trait_rows(metrics(bundle.y[indices], predictions), n=len(indices))

    test = manifest['test']
    test_metrics = trait_rows(metrics(bundle.y[test], prediction), n=len(test))
    history.extend(history_rows(final_info, fold='refit'))
    mixture.extend({'fold': 'refit', **r} for r in final_info['scalar_mix_weights'])
    return write_excel(path, {
        'Ringkasan CV': pd.DataFrame([cv_summary]),
        'CV per Fold dan Trait': pd.DataFrame(cv_fold),
        'CV OOF per Trait': pd.DataFrame(cv_oof),
        'Prediksi OOF': pd.DataFrame(oof_rows),
        'Test per Trait': pd.DataFrame(test_metrics),
        'Prediksi Test': pd.DataFrame(prediction_rows(bundle, test, prediction)),
        'Riwayat Training': pd.DataFrame(history),
        'Scalar Mix': pd.DataFrame(mixture),
    }, {
        **bundle.summary, 'eksperimen': 'E4', 'dataset_version': cfg['dataset_version'],
        'development_rows': len(manifest['development']), 'test_rows': len(test),
        'split_sha256': manifest['split_sha256'], 'notebook_code_sha256': NOTEBOOK_CODE_SHA256,
        'test_dipakai_untuk_seleksi_epoch': False, 'final_config': cfg,
        'kriteria_early_stopping': 'Akar rata-rata kuadrat error seluruh skor validasi dalam satuan asli.',
    })


### Jalankan evaluasi test

In [ ]:
test_prediction = evaluate_test(refit_result, bundle, manifest, bge, cfg, output)
test_metrics = metrics(bundle.y[manifest['test']], test_prediction)
display(pd.DataFrame(trait_rows(test_metrics)))
print(f"Jawaban test dievaluasi: {len(test_prediction)}")


## 19 · Susun Excel hasil

Gabungkan hasil CV, prediksi out-of-fold, hasil test, dan riwayat training yang sudah tersedia.

In [ ]:
report_path = output / 'Hasil_AES_E4_S3_MultiTrait.xlsx'
sheets = export_results(report_path, bundle, manifest, cfg, cv_summary, folds, final_info, test_prediction)
display(pd.DataFrame.from_dict(sheets, orient='index').rename_axis('sheet'))
print(f'Excel hasil: {report_path}')


## 20 · Unduh Excel

In [ ]:
from google.colab import files
files.download(str(report_path))
